# 02 - Labels, QC (Table 2), temporal profile, KEV stability, seal the splits
No model is trained here. The lock at the end freezes `config.lockable()`.

In [ ]:
# Colab setup: repository at /content/kev-text (git clone or unzip), then run all.
import os, sys
REPO = "/content/kev-text"
if os.path.exists(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
# !pip install -q -r requirements.txt          # once per Colab session
import numpy as np, pandas as pd
from src import config, io_utils, labels, evaluate
SYN = False   # True = dry run on src.synthetic; everything goes to outputs_synthetic/

In [ ]:
from src import validate, sensitivity, plots
df = pd.read_parquet(config.DATASET)
snap = pd.Timestamp(df["kev_snapshot_date"].iloc[0])
rep_raw = validate.qc_report(labels.add_labels(df, snap))     # before exclusions: REJECTED, language
df = df[(df["state"] == "PUBLISHED") & df["lang_ok"] & (df["published_date"] >= "2022-01-01")]
labels.assert_after_kev_launch(df)
df = labels.add_labels(df, snap)
df["has_exploit_phrase"] = df["desc_original"].map(labels.has_exploit_phrase)   # reporting only
rep = validate.qc_report(df)
validate.assert_ready(rep)
{"rejected": rep_raw["rejected"], "no_english": rep_raw["no_english_description"],
 **{k: v for k, v in rep.items() if not hasattr(v, "shape")}}

In [ ]:
rep["restoration_bias"]          # Fisher / chi2 overall and per year (Holm)

In [ ]:
prof = validate.temporal_profile(df)
prof_split = validate.temporal_profile(df, by="split")
evaluate.save_table(prof, "table_temporal_profile", is_synthetic=False)
evaluate.save_table(prof_split, "table_split_profile", is_synthetic=False)
prof_split

In [ ]:
validate.group_report(df, "vendor").head(20)

In [ ]:
validate.group_report(df, "cwe").head(20)

In [ ]:
lab = df[df["label_12m"].notna()]
validate.length_report(lab["desc_original"], lab["label_12m"])   # set MAX_LENGTH in config before the lock

In [ ]:
split0 = labels.assign_split(df["published_date"])
validate.near_duplicate_report(df["desc_original"].fillna(""), split0)

## Figure 8: KEV retroactive curve and the S1 start

In [ ]:
# Use data.kev_union(snapshot files) if day-1 check A passed; else the current catalogue
kev_all = df[["cve_id", "kev_date_added"]].dropna()
m = sensitivity.kev_retroactive_monthly(kev_all, df[["cve_id", "published_date"]])
s1 = sensitivity.s1_record(m, df, split0)
plots.save_figure(plots.kev_retroactive_curve(m, s1["start"]), "fig8_kev_retroactive", None, False)
s1

## Seal once

In [ ]:
val = df[split0 == "val"]
n_pos_val_b = int(val.loc[val["published_date"] > config.VAL_A_END, "label_12m"].sum())
alpha = config.conformal_alpha(n_pos_val_b)
lock = labels.write_lock(df, "initial_seal", "boundaries fixed after counts in 02",
                        extra={"s1": s1, "conformal_alpha": alpha, "n_pos_val_b": n_pos_val_b})
df.to_parquet(config.DATA / "dataset_labelled.parquet")
lock["counts"]